# Granite Speech 4.1 2B Evaluation

This notebook evaluates the IBM Granite-Speech-4.1-2B model on a dataset of audio files using a JSONL manifest.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/evaluate_granite_speech.ipynb)

In [ ]:
# @title Install packages and Setup Workarounds
!pip install -q transformers torchaudio peft soundfile accelerate google-cloud-storage

import builtins


# The Magic Hack: Create a dummy class and inject it into Python's builtins
# so the Python 3.12 type-hint evaluator finds it and stops crashing.
class DummyPeftConfig:
    pass


builtins.PeftConfigLike = DummyPeftConfig

import os
import json
import sys
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from google.cloud import storage

# Import common utils
from common.gcs_utils import download_jsonl_manifest, upload_inference_results
from common.inference_pipeline_runner import run_inference_pipeline
from common.audio_utils import preprocess_audio_for_model

# Configure logging
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# --- Configuration ---
MODEL_NAME = "ibm-granite/granite-speech-4.1-2b"
SELECTED_MODEL_KEY = "granite_speech_2b"

GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>"
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"

BATCH_SIZE = 4
LIMIT = 10

In [ ]:
# @title Load the model and processor
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading Granite Speech model on {device}...")

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    device_map=device,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
)

In [ ]:
# @title Define helper functions for evaluation runner
import torchaudio
import torchaudio.functional as F


def prompt_formatter(entry, local_path):
    """Pass the audio file path."""
    return local_path


def granite_speech_inference(model, prompts):
    """Runs inference using processor and model.generate."""
    audio_list = []
    for audio_path in prompts:
        audio, sr = torchaudio.load(audio_path)
        # Ensure mono
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)
        # Resample to 16000 Hz if necessary
        if sr != 16000:
            audio = F.resample(audio, sr, 16000)
        audio_list.append(audio.squeeze(0).numpy())

    inputs = processor(
        audio_list, sampling_rate=16000, return_tensors="pt", padding=True
    ).to(device)

    # Ensure inputs match the model's expected type if using bfloat16
    if device == "cuda":
        inputs = {
            k: v.to(torch.bfloat16) if v.dtype.is_floating_point else v
            for k, v in inputs.items()
        }

    generated_ids = model.generate(**inputs)
    return generated_ids


def result_decoder(ans, model):
    """Extracts the transcription from model's output."""
    return processor.decode(ans, skip_special_tokens=True)

In [ ]:
# @title Run Evaluation

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_inference_pipeline(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=granite_speech_inference,
    decode_fn=result_decoder,
    preprocess_fn=preprocess_audio_for_model,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=SELECTED_MODEL_KEY,
    batch_size=BATCH_SIZE,
    limit=LIMIT,
)

In [ ]:
# @title Upload Inference Results

# Upload results directly to GCS from memory
gcs_uri = upload_inference_results(
    storage_client,
    GCS_BUCKET,
    PROJECT_NAME,
    SELECTED_MODEL_KEY,
    EXPERIMENT_NAME,
    results_list,
)
print(f"Successfully uploaded inference results to: {gcs_uri}")